### PS1: Food Delivery Pricing Engine (Strategy Pattern)

In [9]:
from abc import ABC, abstractmethod

In [10]:
class PricingStrategy(ABC):
    @abstractmethod
    def calculate_price(self,distance):
        pass

In [11]:
class NormalPricing(PricingStrategy):
    def calculate_price(self,distance):
        return 30+distance*5

In [12]:
class PeakHourPricing(PricingStrategy):
    def calculate_price(self,distance):
        return 50+distance*8

In [13]:
class FestivalPricing(PricingStrategy):
    def calculate_price(self,distance):
        return 40+distance*6

In [14]:
class PremiumCustomerPricing(PricingStrategy):
    def calculate_price(self,distance):
        return max(0,(30+distance*5)-20)

In [15]:
class DeliveryService:
    def __init__(self,strategy):
        self.strategy=strategy
    def set_strategy(self,strategy):
        self.strategy=strategy
    def calculate_bill(self,distance):
        return self.strategy.calculate_price(distance)

In [16]:
delivery=DeliveryService(NormalPricing())
print("Normal:",delivery.calculate_bill(10))
delivery.set_strategy(PeakHourPricing())
print("Peak:",delivery.calculate_bill(10))
delivery.set_strategy(FestivalPricing())
print("Festival:",delivery.calculate_bill(10))
delivery.set_strategy(PremiumCustomerPricing())
print("Premium:",delivery.calculate_bill(10))

Normal: 80
Peak: 130
Festival: 100
Premium: 60


### PS2: ATM Machine(State Pattern)

In [1]:
from abc import ABC, abstractmethod

In [2]:
class ATMState(ABC):
    @abstractmethod
    def insert_card(self,atm):
        pass
    @abstractmethod
    def eject_card(self,atm):
        pass
    @abstractmethod
    def enter_pin(self,atm):
        pass
    @abstractmethod
    def withdraw_cash(self,atm,amount):
        pass

In [3]:
class NoCardState(ATMState):
    def insert_card(self,stm):
        print("Card Inserted")
        atm.state=CardInsertedState()
    def eject_card(self,atm):
        print("No card Present")
    def enter_pin(self,atm):
        print("Insert Card First")
    def withdraw_cash(self,atm,amount):
        print("Insert Card First!")

In [4]:
class CardInsertedState(ATMState):
    def insert_card(self,stm):
        print("Card Already Inserted")
        atm.state=NoCardState()
    def eject_card(self,atm):
        print("Card Enjected!")
    def enter_pin(self,atm):
        print("PIN Verified")
        atm.state=PinVerifiedState()
    def withdraw_cash(self,atm,amount):
        print("Enter Pin First!")

In [5]:
class PinVerifiedState(ATMState):
    def insert_card(self,stm):
        print("Card Already Inserted")
    def eject_card(self,atm):
        print("Card Enjected!")
        atm.state=NoCardState()
    def enter_pin(self,atm):
        print("PIN Already Verified")
        atm.state=PinVerifiedState()
    def withdraw_cash(self,atm,amount):
        if amount>atm.balance:
            print("Insufficient Balance")
            return
        atm.balance-=amount
        left_amt=atm.balance-amount
        print(f"Dispensed rs {amount}")
        print(f"left amount rs {left_amt}")
        atm.state=CashDispensedState()

In [6]:
class CashDispensedState(ATMState):
    def insert_card(self,stm):
        print("Transaction Completed")
    def eject_card(self,atm):
        print("Card Ejected!")
        atm.state=NoCardState()
    def enter_pin(self,atm):
        print("Transaction Completed")
        atm.state=PinVerifiedState()
    def withdraw_cash(self,atm,amount):
        print("Already Dispensed!")

In [7]:
class ATM:
    def __init__(self,balance):
        self.balance=balance
        self.state=NoCardState()
    def insert_card(self):
        self.state.insert_card(self)
    def eject_card(self):
        self.state.eject_card(self)
    def enter_pin(self):
        self.state.enter_pin(self)
    def withdraw_cash(self,amount):
        self.state.withdraw_cash(self,amount)

In [8]:
atm=ATM(50000)
atm.insert_card()
atm.enter_pin()
atm.withdraw_cash(30000)
atm.eject_card()

Card Inserted
PIN Verified
Dispensed rs 30000
left amount rs -10000
Card Ejected!


### Parking Lot Design

In [9]:
from abc import ABC

In [10]:
class Vehicle(ABC):
    def __init__(self,number):
        self.number=number
class Bike(Vehicle):
    pass
class Car(Vehicle):
    pass
class Truck(Vehicle):
    pass

In [11]:
class ParkingSpot:
    def __init__(self,spot_id,vehicle_type):
        self.spot_id=spot_id
        self.vehicle_type=vehicle_type
        self.vehicle=None
    def is_free(self):
        return self.vehicle is None
    def park(self,vehicle):
        if not self.is_free():
            return False
        self.vehicle=vehicle
        return True
    def unpark(self):
        self.vehicle=None

In [12]:
class Floor:
    def __init__(self,floor_no):
        self.floor_no=floor_no
        self.spots=[]
    def add_spot(self,spot):
        self.spots.append(spot)
    def get_free_spot(self,vehicle):
        for spot in self.spots:
            if (
                spot.vehicle_type==vehicle.__class__.__name__ and spot.is_free()
            ):
                return spot
        return None

In [24]:
class Ticket:
    counter=1
    def __init__(self,vehicle,spot):
        self.id=Ticket.counter
        Ticket.counter+=1
        self.vehicle=vehicle
        self.spot=spot

In [27]:
class ParkingLot:
    def __init__(self):
        self.floors=[]
        self.tickets={}
    def add_floor(self,floor):
        self.floors.append(floor)
    def park(self,vehicle):
        for floor in self.floors:
            spot=floor.get_free_spot(vehicle)
            if spot:
                spot.park(vehicle)
                ticket=Ticket(vehicle,spot)
                self.tickets[ticket.id]=ticket
                print(f"Parked at Spot {spot.spot_id}")
        return ticket
    def unpark(self,ticket_id):
        ticket=self.tickets[ticket_id]
        ticket.spot.unpark()
        del self.tickets[ticket_id]
        print("Vehicle Unparked!")

In [28]:
parking=ParkingLot()
floor1=Floor(1)
floor1.add_spot(ParkingSpot(1,"Bike"))
floor1.add_spot(ParkingSpot(2,"Car"))
floor1.add_spot(ParkingSpot(3,"Truck"))
parking.add_floor(floor1)
ticket=parking.park(Car("UP32AB1234"))
parking.unpark(ticket.id)

Parked at Spot 2
Vehicle Unparked!
